In [0]:
# Notebook : gold_dim_date
# Domain: Cross-Domain Reference
# Source: silver.date
# Target: gold.dim_date
# Description: Derives SK_DateID surrogate key from DateValue (yyyyMMdd format),
#              filters to 1950-2020 range, and writes the calendar dimension
#              to the Gold star schema layer. Run_id carry-forwarded from Silver.

In [0]:
import logging
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType

#Initialize logger
logger = logging.getLogger("SilverToGold_DimDate")
logger.setLevel(logging.INFO)

In [0]:
source_silver_table = "charles_schwab_retailbrokerage_dev_team_lemma.silver.date"
target_gold_table = "charles_schwab_retailbrokerage_dev_team_lemma.gold.dim_date"

In [0]:
def build_dim_date():
    logger.info("Silver to gold transformation for dim_date")

    try:
        silver_df = spark.table(source_silver_table)
        #Surrogate key
        gold_df = (
            silver_df
            .withColumn("SK_DateID", date_format(col("DateValue"), "yyyyMMdd").cast(IntegerType()))
            .withColumn("_load_ts", current_timestamp())
        )

        #Schema Columns
        final_gold_df = gold_df.select(
            "SK_DateID",
            "DateValue",
            "DateDesc",
            "CalendarYearID",
            "CalendarYearDesc",
            "CalendarQtrID",
            "CalendarQtrDesc",
            "CalendarMonthID",
            "CalendarMonthDesc",
            "CalendarWeekID",
            "CalendarWeekDesc",
            "DayOfWeekNum",
            "DayOfWeekDesc",
            "FiscalYearID",
            "FiscalQtrID",
            "HolidayFlag",
            "_batch",
            "_load_ts"
        )
        (
            final_gold_df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_gold_table)
        )
        logger.info("Trnsformed and wrote dim_date")
        return True
    except Exception as e:
        logger.error("Pipeline failed during Gold dim_date transformation")
        raise e 

table_built = build_dim_date()

In [0]:
if table_built:
    try:
        gold_df = spark.table(target_gold_table)
        actual_count = gold_df.count()
        expected_count = 25933

        logger.info("Gold Reconciliation Summary")
        logger.info("Target Table : dim_date")
        logger.info(f"Expected Rows : {expected_count}")
        logger.info(f"Actual Rows : {actual_count}")

        if actual_count == expected_count:
            logger.info("Success")
        else:
            logger.warning("Failed")

        display(f"Actual Count :{actual_count}")
        display(f"Expected Count :{expected_count}")

    except Exception as e:
        logger.error("Reconciliation failed")
        raise e